
## Pricing Case Study

You’re launching a ride-hailing service that matches riders with drivers for trips between the Toledo Airport and Downtown Toledo. It’ll be active for only 12 months. You’ve been forced to charge riders $30 for each ride. You can pay drivers what you choose for each individual ride.

The supply pool (“drivers”) is very deep. When a ride is requested, a very large pool of drivers see a notification informing them of the request. They can choose whether or not to accept it. Based on a similar ride-hailing service in the same market, you have some data on which ride requests were accepted and which were not. (The PAY column is what drivers were offered and the ACCEPTED column reflects whether any driver accepted the ride request.)

The demand pool (“riders”) can be acquired at a cost of $30 per rider at any time during the 12 months. There are 10,000 riders in Toledo, but you can’t acquire more than 1,000 in a given month. You start with 0 riders. “Acquisition” means that the rider has downloaded the app and may request rides. Requested rides may or may not be accepted by a driver. In the first month that riders are active, they request rides based on a Poisson distribution where lambda = 1. For each subsequent month, riders request rides based on a Poisson distribution where lambda is the number of rides that they found a match for in the previous month. (As an example, a rider that requests 3 rides in month 1 and finds 2 matches has a lambda of 2 going into month 2.) If a rider finds no matches in a month (which may happen either because they request no rides in the first place based on the Poisson distribution or because they request rides and find no matches), they leave the service and never return.




----

#### Assumptions
- Revenue per rider: $30
- If a rider finds no matches, we lose them forever
- Total pool of riders: 10,000
- Maximum acquired riders in a month: 1,000
----
#### Goal

*Find the optimal price to maximize profit*

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import random
import re
import math
import tqdm
from scipy.stats import poisson,iqr
import seaborn as sns
sns.set()

## Import Data

In [ ]:
url = "https://raw.githubusercontent.com/jingle77/Data-Science-Portfolio/main/Ride%20Pricing/pricing.csv"
df = pd.read_csv(url)

## Optimal Number of Bins

In [ ]:
## https://en.wikipedia.org/wiki/Freedman%E2%80%93Diaconis_rule
x = np.array(df['PAY'])
h = (2*iqr(x)) / (len(x)**(1/3))
num_bins = (max(x) - min(x)) / h
df['MyQuantileBins'] =  pd.qcut(df['PAY'], round(num_bins))

## Creating a Pool of Variables to choose from

In [ ]:
df_sorted = df.sort_values(by='PAY').reset_index(drop=True)
grouped_df = df_sorted.groupby("MyQuantileBins")['ACCEPTED'].agg(['count','mean'])

lower_bound_list = np.array([0,9.373,12.913,15.141,17.472,19.418,20.834,22.298,23.641,24.886,26.208,27.701,29.364])
accept_probs = grouped_df['mean'].values[0:len(lower_bound_list)]
profit = 30 - lower_bound_list

variable_pool = pd.DataFrame({
    "PAY":lower_bound_list,
    "PROBABILITY":accept_probs,
    "PROFIT":profit
})

variable_pool['REVENUE'] = 0

## Create a Pool of Customers

In [ ]:
customer_id = list(range(10000))
current_lambda = [1 for x in range(10000)]
in_service = [1 for x in range(10000)]

POOL = pd.DataFrame({
    "ID":customer_id,
    "LAMBDA":current_lambda,
    "IN_SERVICE":in_service
})

## Simulation

In [ ]:
## Set the stage for the simulation. Sets the acceptance probability with the corresponding profit per ride 
for l in range(13):

  TOTAL_REVENUE = 0
  PROB = variable_pool['PROBABILITY'][l]
  PROFIT = variable_pool['PROFIT'][l]

  ACCEPTANCE = [0,1]
  ACCEPTANCE_WEIGHTS = [(1-PROB),PROB]

  customer_pool = POOL.copy()
  random.seed(77)

  for m in range(13):
    
    ## Random sample of cusomers for current month from pool of 10000
    ## If there is less than 10000 in the pool, then sample the whole pool
    if len(customer_pool) < 1000:
      current_month_customers = customer_pool
      current_month_customers = current_month_customers.reset_index(drop=True)

    else:
      current_month_customers = customer_pool.sample(n=1000)
      current_month_customers = current_month_customers.reset_index(drop=True)

    ## For each customer, determine the # of rides
    for c in np.arange(len(current_month_customers)):
      ID = current_month_customers['ID'][c]
      current_lambda = current_month_customers['LAMBDA'][c]

      ## Generate poisson distribution to determine # of rides
      poisson_probs = [poisson.pmf(x, 1) for x in range(31)]
      poisson_weights = [p * 100 for p in poisson_probs]
      poisson_lambda = [x for x in range(31)]
      num_rides = random.choices(poisson_lambda, weights=poisson_weights, k=1)[0]
      
      ## If 0 rides then drop customer
      if num_rides == 0:
          drop_customer_index = customer_pool[customer_pool['ID'] == ID].index
          customer_pool.drop(drop_customer_index, inplace=True)
      else:

        ## For each ride, simulate the drivers accepting
        for r in range(num_rides):
          driver_accepted = random.choices(ACCEPTANCE, weights=ACCEPTANCE_WEIGHTS, k=1)[0]

          ## If driver accepts, add to revenue
          ## Update Lambda based on the number of rides
          ## If a driver does not accept, delete customer from customer pool
          if driver_accepted == 1:
            TOTAL_REVENUE += PROFIT
            if customer_pool[customer_pool['ID'] == ID].LAMBDA.values[0] != num_rides:
              lambda_index = customer_pool[customer_pool['ID'] == ID].index
              customer_pool['LAMBDA'][lambda_index] = num_rides
          else:
            drop_customer_index = customer_pool[customer_pool['ID'] == ID].index
            customer_pool.drop(drop_customer_index, inplace=True)
            break
  print(TOTAL_REVENUE)
  variable_pool['REVENUE'][l] += TOTAL_REVENUE

6870.0


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


7425.720000000058
1862.482999999999
9509.760000000122
17639.424000000054
11354.486000000215
20458.511999999922
14810.945999999592
34586.60100000074
32939.27400000353
32944.89600000477
16732.122000002946
6332.652000001308


## Optimal Pay

In [ ]:
variable_pool[variable_pool['REVENUE'] == variable_pool['REVENUE'].max()]

,PAY,PROBABILITY,PROFIT,REVENUE
8,23.641,0.528302,6.359,34586
